In [ ]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn import tree
import graphviz
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb

pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [ ]:
flight_data = pd.read_csv('./dataset/merged_data/latter_flight_data.csv')
flight_data.head()
flight_data.shape

In [ ]:
# Drop NaN for latter flight data
if True:
    flight_data.dropna(subset=['FORMER_FLIGHT_STATUS'], inplace=True)

In [ ]:
X = flight_data.drop(columns=['ARR_DELAY'])
y = flight_data['ARR_DELAY']

In [ ]:
potential_categorical_vars = {}
# For categorical variables if we are considering variables which were coded to integers, but are actually catagorical.
if True:
    for col in flight_data.columns:
        unique_col_vals = flight_data[col].unique()
        if(len(unique_col_vals) < 15):
            potential_categorical_vars[col] = unique_col_vals
potential_categorical_vars

In [ ]:
# Handle Categorical Variables
categorical_vars = ['DAY_OF_WEEK', 'MKT_UNIQUE_CARRIER',
                    'OP_UNIQUE_CARRIER', 'ORIGIN',
                    'ORGIN_WTH_precipprob', 'ORGIN_WTH_severerisk',
                    'DEST_WTH_precipprob', 'DEST_WTH_severerisk',
                    'FORMER_FLIGHT_STATUS', 'MONTH']

# categorical_vars = potential_categorical_vars.keys()

In [ ]:
def preprocess(flight_data: pd.DataFrame):

    # Dealing with date and time
    flight_data['SCH_ARR_TIME'] = pd.to_datetime(flight_data['SCH_ARR_TIME'])
    flight_data['SCH_DEP_TIME'] = pd.to_datetime(flight_data['SCH_DEP_TIME'])

    flight_data['MONTH'] = flight_data['SCH_ARR_TIME'].dt.month
    flight_data['DAY'] = flight_data['SCH_ARR_TIME'].dt.day
    flight_data['DEP_MINUTES'] = flight_data['SCH_DEP_TIME'].dt.hour * 60 + flight_data['SCH_DEP_TIME'].dt.minute
    flight_data['ARR_MINUTES'] = flight_data['SCH_ARR_TIME'].dt.hour * 60 + flight_data['SCH_ARR_TIME'].dt.minute

    flight_data.drop(columns=['SCH_DEP_TIME', 'SCH_ARR_TIME'], inplace=True)

    # Dropping unwanted columns
    cols = [
        'ORGIN_WTH_temp', 'DEST_WTH_temp',
        'DEST_WTH_severerisk', 'ORGIN_WTH_severerisk',
        'DEST_WTH_precipprob', 'ORGIN_WTH_precipprob'
        ]
    flight_data.drop(columns=cols, inplace=True)
    
    cat_col = list(set(flight_data.columns).intersection(categorical_vars))
    flight_data = pd.get_dummies(flight_data, columns = list(cat_col), drop_first = False)

    return flight_data
    

In [ ]:
X = preprocess(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state = 35)

X_train.head()
X_test.head()
y_train.head()
y_test.head()


In [ ]:
def sMAPE_metric(actual_values, predicted_values):
    actual_predicted_absolute_sum = np.abs(actual_values) + np.abs(predicted_values)
    actual_predicted_absolute_diff = np.abs(actual_values - predicted_values)
    sMAPE = np.mean(actual_predicted_absolute_diff / actual_predicted_absolute_sum)
    return sMAPE

In [ ]:
gb = GradientBoostingRegressor(random_state=50, min_samples_leaf = 2, min_samples_split = 6, max_depth = 5)

gb = gb.fit(X_train, y_train) 
test_output_gb = pd.DataFrame(gb.predict(X_test), index = X_test.index, columns = ['pred_ARR_DELAY'])
test_output_gb = test_output_gb.merge(y_test, left_index = True, right_index = True)
test_output_gb.head()
mean_absolute_error_gb = abs(test_output_gb['pred_ARR_DELAY'] - test_output_gb['ARR_DELAY']).mean()
print(f"GradientBoostingRegressor mean absolute error is: {mean_absolute_error_gb:.2f}")
print(f"GradientBoostingRegressor error ratio: {abs(test_output_gb['pred_ARR_DELAY'] - test_output_gb['ARR_DELAY']).mean()/test_output_gb['ARR_DELAY'].mean():.2f}")
test_output_gb['ARR_DELAY'].mean()

In [ ]:
print(f"sMAPE for GradientBoostingRegressor model: {sMAPE_metric(test_output_gb['pred_ARR_DELAY'], test_output_gb['ARR_DELAY']) * 100:.2f}")

## Trying classification

In [ ]:
def delay_categories(delay):
    if delay < -7:
        return 0 #early
    elif delay > 6:
        return 2 #late
    else:
        return 1 #ontime

In [ ]:
y = flight_data['ARR_DELAY'].apply(delay_categories)
y.value_counts()

##### Trying SMOT

In [ ]:
import imblearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

if False:
    over = SMOTE()
    under = RandomUnderSampler()
    steps = [('o', over), ('u', under)]
    pipeline = Pipeline(steps=steps)

    x_smote, y_smote = pipeline.fit_resample(X, y)

    counter = Counter(y)
    for k,v in counter.items():
        per = v / len(y) * 100

    plt.bar(counter.keys(), counter.values())
    plt.xlabel("Class value distribution before sampling")
    plt.show()

    counter = Counter(y_smote)
    for k,v in counter.items():
        per = v / len(y) * 100

    plt.bar(counter.keys(), counter.values())
    plt.xlabel("Class value distribution before sampling")
    plt.show()

    X_train, X_test, y_train, y_test = train_test_split(x_smote, y_smote, stratify=y_smote, test_size=0.20, random_state = 21)
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.20, random_state = 21)



In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'min_child_weight': [1, 2, 3],
    'learning_rate': [0.1, 0.01, 0.005],
    'max_depth': [1, 2, 3, 4]
}

xgb_classifier = XGBClassifier(
    n_estimators=600,
    reg_lambda=0.007
)

grid_search = GridSearchCV(
    estimator=xgb_classifier,
    param_grid=param_grid,
    scoring='accuracy',  # Consider other metrics such as 'f1_macro' for imbalanced classes
    cv=7,
    verbose=1
)


grid_search.fit(X, y)


print("Best parameters for the XGBClassifier model:", grid_search.best_params_)
best_score = grid_search.best_score_ * 100
print(f"Best Accuracy: {best_score:.4f}%")


for i in range(len(grid_search.cv_results_['mean_test_score'])):
    params = grid_search.cv_results_['params'][i]
    mean_test_score = grid_search.cv_results_['mean_test_score'][i]
    std_deviation_score = grid_search.cv_results_['std_test_score'][i]
    print(f"Parameters selected: {params}, Mean Accuracy: {mean_test_score*100:.4f}%, Standard Deviation: {std_deviation_score*100:.4f}%")


In [ ]:
def lineplot(learning_rate_range, train, test, title):
    plt.plot(learning_rate_range, test, c='m', label='Test')
    plt.plot(learning_rate_range, train, c='orange', label='Train')
    plt.xlabel('Learning rate')
    plt.xticks(learning_rate_range)
    plt.ylabel('Accuracy score')
    plt.ylim(0.5, 1)
    plt.legend(prop={'size': 12}, loc=3)
    plt.title(title, size=14)

#### xgBoost

In [ ]:
if False:
    # XGBoost (different learning rate)
    learning_rate_range = np.arange(0.01, 1, 0.05)
    test = [] 
    train = []
    for lr in learning_rate_range:
        xgb_classifier = xgb.XGBClassifier(eta = lr)
        _ =xgb_classifier.fit(X_train, y_train)
        train.append(xgb_classifier.score(X_train, y_train))
        test.append(xgb_classifier.score(X_test, y_test))
    # Line plot
    fig = plt.figure(figsize=(10, 7))
    lineplot(learning_rate_range, train, test, title='Accuracy score vs. Learning rate of XGBoost')
    plt.show()

    # Resolve overfitting 
    # new learning rate range
    learning_rate_range = np.arange(0.01, 0.5, 0.05)
    fig = plt.figure(figsize=(19, 17))
    idx = 1
    # grid search for min_child_weight
    for weight in np.arange(0, 4.5, 0.5):
        train = []
        test = []
        for lr in learning_rate_range:
            xgb_classifier = xgb.XGBClassifier(eta = lr, reg_lambda=1, min_child_weight=weight)
            _ = xgb_classifier.fit(X_train, y_train)
            train.append(xgb_classifier.score(X_train, y_train))
            test.append(xgb_classifier.score(X_test, y_test))
        fig.add_subplot(3, 3, idx)
        idx += 1
        title = "Min child weight:" + str(weight)
        lineplot(learning_rate_range, train ,test, title)
    plt.show()

In [ ]:
xg_model = xgb.XGBClassifier(eta = '0.08', max_depth=5, min_child_weight=1, reg_lambda=0.007)
xg_model = xg_model.fit(X_train, y_train) 
xg_model.score(X_train, y_train) 

# gb.feature_importances_
feat_imp = pd.Series(xg_model.feature_importances_, X_train.columns.values).sort_values(ascending=False)

feat_imp_table = pd.DataFrame(feat_imp)
feat_imp_table = feat_imp_table.reset_index()
feat_imp_table.columns = ['Features', 'Values']
feat_imp.head(20)

test_output = pd.DataFrame(xg_model.predict(X_test), index = X_test.index, columns = ['pred_Y'])
test_output = test_output.merge(y_test, left_index = True, right_index = True)
test_output.head()
print('Fraction of correct classification ')
xg_model.score(X_test, y_test) 
test_output['pred_Y'].value_counts()

#### GradientBoosting

In [ ]:
gb = GradientBoostingClassifier(random_state=50, min_samples_split = 2, min_samples_leaf = 6, max_depth = 7, n_estimators = 200)
gb = gb.fit(X_train, y_train) 
gb.score(X_train, y_train) 

# gb.feature_importances_
feat_imp = pd.Series(gb.feature_importances_, X_train.columns.values).sort_values(ascending=False)

feat_imp_table = pd.DataFrame(feat_imp)
feat_imp_table = feat_imp_table.reset_index()
feat_imp_table.columns = ['Features', 'Values']
feat_imp.head(20)

test_output = pd.DataFrame(gb.predict(X_test), index = X_test.index, columns = ['pred_Y'])
test_output = test_output.merge(y_test, left_index = True, right_index = True)
test_output.head()
print('Fraction of correct classification ')
gb.score(X_test, y_test) 
test_output['pred_Y'].value_counts()

##### Random Forest

In [ ]:
rf = RandomForestClassifier(max_features='sqrt', min_samples_leaf=4)
rf = rf.fit(X_train, y_train)
rf.score(X_train, y_train)

feat_imp = pd.Series(rf.feature_importances_, X_train.columns.values).sort_values(ascending=False)
feat_imp.head(20)

test_output = pd.DataFrame(rf.predict(X_test), index = X_test.index, columns = ['pred_Y'])

test_output.head()
test_output = test_output.merge(y_test, left_index = True, right_index = True)
test_output.head()
print('Fraction of correct classification ')
rf.score(X_test, y_test) 
test_output['pred_Y'].value_counts()